In [16]:
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage,ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from IPython.display import Image, display
from typing import Literal
from ddgs import DDGS
import random
import os

print("♻ Libraries import successful")

♻ Libraries import successful


In [21]:
load_dotenv
open_api_key = os.getenv("OPENAI_API_KEY")

if not open_api_key:
    raise ValueError("Openai key not found")
print("🔑 API Key Loaded")

🔑 API Key Loaded


In [22]:
# Lets initialize LLM
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature = 2,
    api_key=open_api_key
)

print(f"♻ LLM initialized: {llm.model_name}")

♻ LLM initialized: gpt-4o-mini


In [23]:

@tool
def weather(city: str) -> str:
    """
    Return a simulated weather for a given city
    """
    weather_conditions = ["Sunny", "Cloudy", "Rainy", "Thunderstorm", "Windy"]
   
    try:
        temperature = random.randint(18, 35)
        condition = random.choice(weather_conditions)
        
        result = f"The current weather in {city} is {condition} with a temperature of {temperature}°C."
        return str(result)
    except Exception as e:
        return f"Error checking weather: {str(e)} "
print("☁🌡 weather tool created")

☁🌡 weather tool created


In [24]:
# Lets Test the tool
result = weather.invoke("Lagos")
print(result)


The current weather in Lagos is Windy with a temperature of 30°C.


In [10]:
from langchain.tools import tool

@tool
def dictionary(word: str) -> str:
    """
    Looks up the definition of a word from a simulated dictionary.
    """
    dictionary = {
        "agent": "An entity that acts on behalf of another to perform tasks.",
        "chatbot": "A computer program designed to simulate conversation with humans.",
        "algorithm": "A step-by-step procedure for solving a problem.",
        "python": "A high-level programming language known for its readability.",
        "tool": "A function an agent can call to perform a specific task."
    }
    try:
        definition = dictionary.get(word.lower())
        
        if definition:
            return f"{word.capitalize()}: {definition}"
        else:
            return f"Sorry, I don't have a definition for '{word}'."
    except Exception as e:
        return f"error looking up word :{str(e)}"
print("📚 Dictionary tool created sucessfully")

📚 Dictionary tool created sucessfully


In [25]:
# Lets Test the tool
result = dictionary.invoke("agent")
print(result)


Agent: An entity that acts on behalf of another to perform tasks.


In [26]:



@tool
def web_search(query: str) -> str:
    """
    Uses DuckDuckGo to search the web for information.
    """
    results = []

    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=3):
            results.append(
                f"- {r['title']}\n  {r['href']}\n  {r['body']}"
            )

    if not results:
        return "No results found."

    return "\n\n".join(results)
print("🕸 web search tool successfully created")


🕸 web search tool successfully created


In [27]:
web_search.invoke("What is LangGraph?")


"- What is LangGraph? - IBM\n  https://www.ibm.com/think/topics/langgraph\n  LangGraph , created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalable and efficient manner. At its core, LangGraph uses the power of graph-based architectures to model and manage the ...\n\n- LangGraph - LangChain\n  https://www.langchain.com/langgraph\n  LangGraph's low-level primitives provide the flexibility needed to create fully customizable agents. Design diverse control flows — single, multi-agent, hierarchical — all using one framework. LangGraph's built-in memory stores conversation histories and maintains context over time, enabling rich, personalized interactions across sessions.\n\n- What is LangGraph? - GeeksforGeeks\n  https://www.geeksforgeeks.org/machine-learning/what-is-langgraph/\n  LangGr

### Binding tools into LLM

In [28]:
# Create list of tools
tools = [weather, dictionary,web_search]

# Bind tools to the llm
llm_with_tools = llm.bind_tools(tools)

print(f"LLM bound to {len(tools)} tools")
print(f" Tools: {[tool.name for tool in tools]}")

LLM bound to 3 tools
 Tools: ['weather', 'dictionary', 'web_search']


In [29]:
response = llm_with_tools.invoke([HumanMessage(content = "What is the weather in Abuja?")])

print(f"Response type: {type(response)}")
print(f"\nContent: {response.content}")
print(f"\nTool calls: {response.tool_calls}")

C:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\typing.py:463: ResourceWarning: unclosed <ssl.SSLSocket fd=1404, family=2, type=1, proto=0, laddr=('192.168.43.172', 60802), raddr=('40.114.177.156', 443)>
  def _eval_type(t, globalns, localns, type_params=_sentinel, *, recursive_guard=frozenset()):


Response type: <class 'langchain_core.messages.ai.AIMessage'>

Content: 

Tool calls: [{'name': 'weather', 'args': {'city': 'Abuja'}, 'id': 'call_RGhwCnpM7oEdNcfqarGX8PGy', 'type': 'tool_call'}]


In [30]:
# System prompt that encourages tool usage
sys_msg = SystemMessage(content="""You are a helpful assistant with access to tools.

When asked to perform calculations, use the calculator tool.
When asked to analyze text, use the text_analyzer tool.

Only use tools when necessary - for simple questions, answer directly.""")

def assistant(state: MessagesState) -> dict:
    """
    Assistant node - decides whether to use tools or answer directly.
    """
    messages = [sys_msg] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

print("✅ Assistant node defined")

✅ Assistant node defined


In [31]:
def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """
    Decide next step based on last message.
    
    If LLM called a tool → go to 'tools' node
    If LLM provided final answer → go to END
    """
    last_message = state["messages"][-1]
    
    # Check if LLM made tool calls
    if last_message.tool_calls:
        return "tools"
    
    # No tool calls - we're done
    return "__end__"

print("✅ Conditional routing function defined")

✅ Conditional routing function defined


In [32]:
# Create the graph
builder = StateGraph(MessagesState)

# Add nodes
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))  # ToolNode executes tool calls automatically

# Define edges
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    should_continue,
    {"tools": "tools", "__end__": END}
)
builder.add_edge("tools", "assistant")  # After tools, go back to assistant

# Add memory
memory = MemorySaver()
agent = builder.compile(checkpointer=memory)

print("✅ Agent graph compiled with tools and memory")

✅ Agent graph compiled with tools and memory


In [33]:
# Visualize the agent graph
try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not display graph: {e}")
    print("Graph structure: START → assistant → [conditional] → tools → assistant → END")

Could not display graph: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`
Graph structure: START → assistant → [conditional] → tools → assistant → END


In [34]:
# Helper function
def run_agent(user_input: str, thread_id: str = "test_session"):
    """
    Run the agent and display the conversation.
    """
    print(f"\n{'='*70}")
    print(f"👤 User: {user_input}")
    print(f"{'='*70}\n")
    
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config={"configurable": {"thread_id": thread_id}}
    )
    
    for message in result["messages"]:
        if isinstance(message, HumanMessage):
            continue  # Already printed
        elif isinstance(message, AIMessage):
            if message.tool_calls:
                print(f"🤖 Agent: [Calling tool: {message.tool_calls[0]['name']}]")
            else:
                print(f"🤖 Agent: {message.content}")
        elif isinstance(message, ToolMessage):
            print(f"🔧 Tool Result: {message.content[:100]}..." if len(message.content) > 100 else f"🔧 Tool Result: {message.content}")
    
    print(f"\n{'='*70}\n")

print("✅ Test function ready")

✅ Test function ready


In [35]:
run_agent("What is the weather in Abeokuta")


👤 User: What is the weather in Abeokuta

🤖 Agent: [Calling tool: weather]
🔧 Tool Result: The current weather in Abeokuta is Thunderstorm with a temperature of 24°C.
🤖 Agent: The current weather in Abeokuta is thundery with a temperature of 24°C.


